# D1 · Reading the coefficients: odds ratios, collinearity and omitted variables

**Type**: Interpret · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

Interpret the coefficients of a logistic correctly: tell odds from probability, measure collinearity (correlation and VIF), see
how the same coefficient changes **sign and size** depending on the other variables in the model, and verify it with a bootstrap interval.

## Dataset

`ds.loans(with_missing=False)` with `ratio = amount/income`, `age`, `job_tenure` and `unemployed` (1 if `contract_type` is "unemployed"). The features are
standardized inside `coefficients(columns, d, yy)` (given), which returns the coefficients as a `pd.Series` (logistic with `C=1e4`, almost unregularized).

## Educational purpose

Dismantle the reflex "the coefficient is the effect of the variable". A coefficient is the effect **all else being equal among the variables in the model**: change the model, the coefficient changes.
You will learn to see the signals (correlation, VIF), to verify the stability (bootstrap) and to suspect an omitted variable.

## Task

1. Compute `corr_age_tenure` (Pearson correlation between `age` and `job_tenure`) and `vif` (`pd.Series` with the variance inflation factor of each of the three features `ratio, age, job_tenure`: $1/(1-R^2)$ where $R^2$ comes from the linear regression of that feature on the other two).
2. With the function `coefficients` compute `coef_a` (columns `ratio, age, job_tenure`), `coef_b` (`ratio, age`) and `coef_c` (`ratio, job_tenure`). Answer with two booleans: `age_matters_alone` (does the coefficient of `age` in `coef_b` exceed 0.1 in absolute value?) and `age_matters_given_tenure` (in `coef_a` does it exceed 0.25?).
3. **Bootstrap** (200 resamplings with `rng = np.random.default_rng(0)`, indices `rng.integers(0, n, n)`): compute the 95% interval (2.5 and 97.5 percentiles) of the coefficient of `age` in model A (`interval_a`) and in model B (`interval_b`), as tuples `(low, high)`.
4. **Omitted variable.** Add `unemployed` to model A: save `coef_e`. By how much (in absolute value) does the coefficient of `age` and the one of `job_tenure` shrink compared to A? Save the two ratios `|coef_e| / |coef_a|` in `shrinkage` (dictionary with keys `age` and `job_tenure`).
5. **Odds versus probability.** With model A, for an average customer (all features standardized to 0) save `base_prob` (default probability) and `after_prob` (after +1 std dev of `ratio`), `ratio_or` (odds ratio of `ratio`) and `prob_delta = after_prob - base_prob`. Check that the odds get multiplied by `ratio_or` but the probability does **not** get multiplied.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False)
df["ratio"] = df["amount"] / df["income"]
df["unemployed"] = (df["contract_type"] == "unemployed").astype(int)
y = df["defaulted"].to_numpy()


def coefficients(columns, d=df, yy=y):
    """Coefficients (on standardized features) of an almost unregularized logistic."""
    m = make_pipeline(StandardScaler(), LogisticRegression(C=1e4, max_iter=2000)).fit(d[columns], yy)
    return pd.Series(m[-1].coef_[0], index=columns)


cols_a = ["ratio", "age", "job_tenure"]
print(f"{len(df)} customers, {y.mean():.1%} defaulters")

## Your work

### 1. Collinearity

In [ ]:
corr_age_tenure = None
vif = None

### 2. The same coefficient in three models

In [ ]:
coef_a = None
coef_b = None
coef_c = None
age_matters_alone = None
age_matters_given_tenure = None

### 3. Bootstrap

In [ ]:
interval_a = None
interval_b = None

### 4. An omitted variable

In [ ]:
coef_e = None
shrinkage = None

### 5. Odds versus probability

In [ ]:
model_a = make_pipeline(StandardScaler(), LogisticRegression(C=1e4, max_iter=2000)).fit(df[cols_a], y)
base_prob = None
after_prob = None
ratio_or = None
prob_delta = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`vif[c] = 1 / (1 - LinearRegression().fit(df[others], df[c]).score(df[others], df[c]))`. Bootstrap: `i = rng.integers(0, n, n); coefficients(cols, df.iloc[i], y[i])`. Average customer: `z0 = model_a[-1].intercept_[0]`; `base_prob = 1 / (1 + np.exp(-z0))`; `z1 = z0 + model_a[-1].coef_[0][0]`.'):
    check(0.8 < corr_age_tenure < 0.9, "age and job_tenure are highly correlated (about 0.85)", "np.corrcoef or df.corr()")
    check(vif is not None and vif["ratio"] < 1.5 and vif["age"] > 3 and vif["job_tenure"] > 3, "the VIF flags the collinearity between age and tenure (above 3), not for ratio", "VIF = 1 / (1 - R^2) from the linear regression of each feature on the other two")
    check(close(coef_a["age"], coefficients(cols_a)["age"]) and close(coef_b["age"], coefficients(["ratio", "age"])["age"]), "coefficients of the three models computed with the given function", "coefficients(columns)")
    check(age_matters_alone is False and age_matters_given_tenure is True, "alone age does not matter (coef close to 0); given tenure it does, and with a positive sign", "compare |coef_b['age']| with 0.1 and coef_a['age'] with 0.25")
    check(interval_a[0] > 0, "in model A the bootstrap interval of age lies entirely above zero", "2.5 and 97.5 percentiles of the 200 coefficients")
    check(interval_b[0] < 0 < interval_b[1], "in model B the interval of age contains zero", "age alone has no effect distinguishable from zero")
    check(shrinkage["age"] < 0.5 and shrinkage["job_tenure"] < 0.5, "by adding `unemployed`, both coefficients shrink to less than half", "|coef_e| / |coef_a| for age and job_tenure")
    check(close((after_prob / (1 - after_prob)) / (base_prob / (1 - base_prob)), ratio_or) and prob_delta < (ratio_or - 1) * base_prob,
          "the odds get multiplied by the odds ratio; the probability goes up far less than 'x times'", "prob = sigmoid(intercept) for the average customer; after_prob = sigmoid(intercept + coefficient of ratio)")
done()

---
The commented solution is in `solutions/D_interpret/D1_coefficients_odds_ratios_and_omitted_variables.ipynb`. Open it after you pass the check, or after you have really tried.